> **Design note:** This notebook covers a convention rather than a transferable concept — the rule is simple enough to state in one sentence, and you only encounter it reactively when CI fails. Per [DESIGN.md](../DESIGN.md), that profile fits a reference doc better than an exercise. The notebook exists for onboarding contexts where the interactive format may be useful, but is not a model for future exercises in this series.

# CI Failure Discovery

**What you'll learn:**
- How to find the real failure in a noisy CI log
- Why the NameError points at a line you didn't write
- How to trace from the error back to the missing guard

**Prerequisites:** complete notebooks 1 and 2 first.

**Time:** ~20 minutes.

In [ ]:
# Setup — run this first.
# Requires the 'passagemath (ci-failure-discovery)' kernel.
import sys
import re
assert sys.version_info >= (3, 11), (
    f"Wrong kernel — expected Python 3.11+, got {sys.version_info}"
)

def simulate_doctest(src, available=frozenset()):
    """
    Label each 'sage:' line as RUN, SKIP, or BLOCK.
    Does not execute code — parses guard syntax only.
    """
    block_needs = None
    for raw in src.strip().split('\n'):
        line = raw.strip()
        if not line.startswith('sage:'):
            continue
        bm = re.match(r'sage:\s*#\s*needs\s+(\S+)\s*$', line)
        if bm:
            block_needs = bm.group(1)
            tag = 'BLOCK' if block_needs not in available else 'BLOCK(run)'
            print(f"  {tag:<12}  {line}")
            continue
        im = re.search(r'#\s*needs\s+(\S+)', line)
        needed = im.group(1) if im else block_needs
        tag = 'SKIP' if (needed and needed not in available) else 'RUN'
        print(f"  {tag:<12}  {line}")

print("Setup OK.")

---
## Section 1: The log

Your PR is up. Test-mod CI failed. Run the cell.

In [ ]:
_log = """\
Step 12/23 : RUN python -m sage.doctest --initial --stats-path=stats.json ...
 ---> Running in c7f3a2b1e4d9
[172.18.0.1] Pulling layer sha256:9a3f7b2c...
Successfully installed passagemath-rings-10.5.27
/usr/lib/python3.11/subprocess.py:1026: DeprecationWarning: This process ...
Running doctests with ID 20260315-1042...
...............................................
----------------------------------------------------------------------
Comparing against known-test-failures.json
  ntests: 47
  failures in baseline: 0

New failures, not in baseline:

sage/rings/polynomial/polynomial_quotient_ring.py
  Failed example:
      _A(_f)
  Exception raised:
      Traceback (most recent call last):
        ...
      NameError: name '_A' is not defined

No other failures.
"""

print(_log)

`ntests: 47` is tests *run*, not failed. The count that matters is under `New failures, not in baseline`. Run the cell to extract it.

In [ ]:
# Same as: grep -A 20 'New failures, not in baseline'
_lines = _log.splitlines()
for _i, _line in enumerate(_lines):
    if 'New failures' in _line:
        print('\n'.join(_lines[_i:_i + 12]))
        break

---
## Section 2: The line you didn't write

One failure. `polynomial_quotient_ring.py`. `NameError: name '_A' is not defined`.

The failing line is `_A(_f)`. It's been in that file for years — it's not in your diff.

The line directly above it assigns `_A`:

```
sage: _R = ZZ['x']; _f = _R('x'); _A = _R.quotient(_f^2)   # needs sage.modules
sage: _A(_f)
xbar
```

`_A` is right there. Run the cell.

In [ ]:
_doctest = """
    sage: _R = ZZ['x']; _f = _R('x'); _A = _R.quotient(_f^2)   # needs sage.modules
    sage: _A(_f)
    xbar
"""

print('=== test-mod CI (sage.modules absent) ===')
simulate_doctest(_doctest, available=set())

The setup line is SKIP. `_A` was never bound. The next line ran anyway and hit an undefined name.

Python doesn't error when a guarded line is skipped — it errors when the unbound name is first *used*. The traceback points at the use. The cause is somewhere earlier.

Why was the setup skipped? `test-mod` installs only the target module's minimal dependencies. For `polynomial_quotient_ring.py` that's `passagemath-rings`, which doesn't include `passagemath-modules`. So `sage.modules` is absent and the `# needs sage.modules` guard fires.

Your local venv has more packages. Run the cell.

In [ ]:
_doctest = """
    sage: _R = ZZ['x']; _f = _R('x'); _A = _R.quotient(_f^2)   # needs sage.modules
    sage: _A(_f)
    xbar
"""

print('=== local venv (sage.modules present) ===')
simulate_doctest(_doctest, available={'sage.modules'})

Both RUN locally. You never see the failure because `sage.modules` is always there.

---
## Section 3: The fix

The setup line has `# needs sage.modules`. The usage line doesn't. Add the guard to `_A(_f)` in the cell below.

Goal: `sage.modules` absent → both lines SKIP.

In [ ]:
_to_fix = """
    sage: _R = ZZ['x']; _f = _R('x'); _A = _R.quotient(_f^2)   # needs sage.modules
    sage: _A(_f)
    xbar
"""

# Add the missing guard above. Goal: both lines SKIP when sage.modules is absent.
print('=== sage.modules absent ===')
simulate_doctest(_to_fix, available=set())

---
### Fix — PR [#2293](https://github.com/passagemath/passagemath/pull/2293)

```diff
-    sage: _A(_f)
+    sage: _A(_f)                                                 # needs sage.modules
```

One line. The guard on the usage matches the guard on the setup.

In [ ]:
_before = """
    sage: _R = ZZ['x']; _f = _R('x'); _A = _R.quotient(_f^2)   # needs sage.modules
    sage: _A(_f)
    xbar
"""

_after = """
    sage: _R = ZZ['x']; _f = _R('x'); _A = _R.quotient(_f^2)   # needs sage.modules
    sage: _A(_f)                                                 # needs sage.modules
    xbar
"""

print('--- Before: sage.modules absent ---')
simulate_doctest(_before, available=set())
print()
print('--- After: sage.modules absent ---')
simulate_doctest(_after, available=set())

---
## Section 4: Exercise — `automatic_semigroup.py`

PR [#2161](https://github.com/passagemath/passagemath/pull/2161) added two rename calls without guards. Test-mod caught them (issue [#2256](https://github.com/passagemath/passagemath/issues/2256)):

```
New failures, not in baseline:

sage/monoids/automatic_semigroup.py
  Failed example:
      S5.rename('S5')
  Exception raised:
      NameError: name 'S5' is not defined

sage/monoids/automatic_semigroup.py
  Failed example:
      M.rename('M(S5)')
  Exception raised:
      NameError: name 'M' is not defined
```

The doctest block:

```
sage: S5 = SymmetricGroup(5)                                   # needs sage.groups
sage: M = AutomaticSemigroup(Family({1: S5((1,2))}),           # needs sage.groups
...  (output)
sage: S5.rename('S5')       <- failing
sage: M.rename('M(S5)')     <- failing
```

Add guards to the exercise cell. Goal: `sage.groups` absent → all four lines SKIP.

In [ ]:
_exercise = """
    sage: S5 = SymmetricGroup(5)                                   # needs sage.groups
    sage: M = AutomaticSemigroup(Family({1: S5((1,2))}),           # needs sage.groups
    sage: S5.rename('S5')
    sage: M.rename('M(S5)')
"""

# Add guards to the two failing lines above.
print('=== sage.groups absent ===')
simulate_doctest(_exercise, available=set())
print()
print('=== sage.groups present ===')
simulate_doctest(_exercise, available={'sage.groups'})

When `sage.groups` absent shows all four lines as SKIP, you've got it.

---
### Answer — issue [#2256](https://github.com/passagemath/passagemath/issues/2256)

```diff
-    sage: S5.rename('S5')
+    sage: S5.rename('S5')          # needs sage.groups
-    sage: M.rename('M(S5)')
+    sage: M.rename('M(S5)')        # needs sage.groups
```

PR #2161 added the rename calls but didn't check whether the setup lines were guarded. Passed locally because `sage.groups` was installed. Test-mod exposed the gap.

---
## Summary

**Discovery path:**
1. Grep the CI log for `New failures, not in baseline` — ignore everything before it
2. Read the NameError: which name, which file
3. Find that name's assignment in the doctest — read its guard
4. Find downstream lines that use the name without that guard
5. Add the guard — syntax in [notebook 2](../needs-sage-guards/needs-sage-guards.ipynb)

**Two traps:**
- `ntests` is tests *run*, not failed
- The NameError points at the usage site — search backward for the cause